# Using LLMs as High-Level Planners for Multi-Agent Coordination

This notebook provides a step-by-step guide to customizing and interacting with the RL environment.

## For Submission
1. Fill in your code in `submit.py`. 
   - Add your code *only* in the TODO sections marked by the '#' delimiter lines. Do not modify any other parts of the script.
   - You should implement any helper functions/classes in a separate `helper.py` file and import them in `submit.py`.
1. Submit `out.log` and `results.csv` generated by the `submit.py` script.


In [1]:
# Import necessary libraries and modules
import gymnasium as gym
import multigrid.envs
import matplotlib.pyplot as plt
from agents import AgentCollection

%matplotlib inline
%load_ext autoreload
%autoreload 2

---
## Initial Plan Generation

An intial plan can be generated by the `initial_planner` by invoking it with the grid size and number of agents available.

In [2]:
from models import local_llm as llm
# from planner import PromptPlanner
from planner import HybridPlanner

In [3]:
def parse_response(env, observations, rewards, agents,terminations, truncations, infos, num_agents):
    messages = ""

    messages += "After Step "+ str(env.unwrapped.step_count) +": \n"
    messages += "Mission:" + str(observations[0]['mission']) + "\n"
    messages += "Num of agent(s):" +  str(num_agents) + "\n"
    messages += "Num of remaining target(s):" + str(observations['global']['num_goals']) + "\n"
    for i in range(num_agents):
        loc = tuple(int(v) for v in (observations[i]['location']))
        messages += "Agent " + str(i) + "'s Location: " + str(loc) + "\n"
        messages += "Is Agent " + str(i) +" idling?: " + str(agents.idle(i)) + "\n"
        messages += "Agent " + str(i) + "'s reward:" +  str(rewards[i])+ "\n"
    messages += "Collective reward for current timestep:" + str(infos['cur_reward']) + "\n"
    messages += "Cumulative reward:"+ str(infos['total_reward']) + "\n"
    messages += "Terminations:" + str(env.unwrapped.is_done()) +  str(terminations) + "\n"
    messages += "Truncations:" + str(truncations) + "\n"
    
    return messages

In [4]:
# N, M = 50, 3
# env = multigrid.envs.EmptyEnvV2(
#     size=N,  # Specify the size of the grid, N
#     agents=M,  # Specify number of agents, M
#     goals=[(5, 5), (45, 45), (20, 20)],  # Specify target positions for agents
#     mission_space="One target is contained within the region from (1, 1) to (6, 6), one target is contained within the region from (16,16) to (21,21) and the other target is contained within the region from (40, 40) to (46, 46).",
#     render_mode="rgb_array",
#     hidden_goals=True,
#     # max_steps=50, # For debugging, you can set a maximum number of steps
# )

In [5]:
N, M = 20, 2

number_of_agents = 2
grid_size = 20
mission_statement = "One target is contained within the region from (1, 1) to (5, 5) and the other target is contained within the region from (10, 10) to (16, 16)."
goals = [(3, 3), (15, 15)]
number_of_trials = 2

env = multigrid.envs.EmptyEnvV2(
    size=N,  # Specify the size of the grid, N
    agents=M,  # Specify number of agents, M
    goals=[(3, 3), (15, 15)],  # Specify target positions for agents
    mission_space=mission_statement,
    render_mode="rgb_array",
    hidden_goals=True,
    # max_steps=50, # For debugging, you can set a maximum number of steps
)

In [6]:
# Always reset the environment before starting
observations, infos = env.reset()

# Create a group of 2 agents
agents = AgentCollection(num=M)

# planner = PromptPlanner(llm=llm, grid_size=N, observations=observations, infos=infos)
planner = HybridPlanner(llm=llm, grid_size=N, observations=observations, infos=infos)

# Providing the agents with high-level instructions
mission = observations[0]["mission"]
plan = planner.initial_plan()
for agent, actions in plan.items():
    for action in actions:
        agents.tell({agent: action.serialize()})

while not agents.all_idle() and not env.unwrapped.is_done():
    # Obtain the low-level action for current time step for all agents
    a = agents.act()

    # Step the environment with the actions
    observations, rewards, terminations, truncations, infos = env.step(a)
    msg = parse_response(env, observations, rewards, agents,terminations, truncations, infos, M)
    # print("------------")
    # print(msg)
    # print(observations)
    # print(a, rewards, terminations, truncations)

    plan = planner.replan(observations, rewards, terminations, truncations, infos)
    for agent, actions in plan.items():
        for action in actions:
            agents.tell({agent: action.serialize()})

    # Render the environment
    # img = env.render()
    # plt.figure(figsize=(5, 5))
    # plt.imshow(img)
    # plt.show()

env.close()

/mount/home/cluoqi/hierarchical_planning/planner/hybrid_planner.py:46: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(




> Entering new AgentExecutor chain...
{
  "agents": {
    "0": [
      {
        "type": "move",
        "cur_x": 1,
        "cur_y": 1,
        "tar_x": 10,
        "tar_y": 10
      },
      {
        "type": "search",
        "cur_x": 10,
        "cur_y": 10,
        "x1": 1,
        "y1": 1,
        "x2": 10,
        "y2": 10
      }
    ],
    "1": [
      {
        "type": "move",
        "cur_x": 1,
        "cur_y": 1,
        "tar_x": 15,
        "tar_y": 15
      },
      {
        "type": "search",
        "cur_x": 15,
        "cur_y": 15,
        "x1": 11,
        "y1": 11,
        "x2": 20,
        "y2": 20
      }
    ]
  }
}

> Finished chain.


In [7]:
N, M = 50, 3
env = multigrid.envs.EmptyEnvV2(
    size=N,  # Specify the size of the grid, N
    agents=M,  # Specify number of agents, M
    goals=[(5, 5), (45, 45), (20, 20)],  # Specify target positions for agents
    mission_space="One target is contained within the region from (1, 1) to (6, 6), one target is contained within the region from (16,16) to (21,21) and the other target is contained within the region from (40, 40) to (46, 46).",
    render_mode="rgb_array",
    hidden_goals=True,
    # max_steps=50, # For debugging, you can set a maximum number of steps
)

# Always reset the environment before starting
observations, infos = env.reset()
print(observations)

{0: {'image': array([[[1, 0, 0]]]), 'direction': np.int64(0), 'mission': Mission("One target is contained within the region from (1, 1) to (6, 6), one target is contained within the region from (16,16) to (21,21) and the other target is contained within the region from (40, 40) to (46, 46)."), 'location': (np.int64(1), np.int64(1))}, 1: {'image': array([[[1, 0, 0]]]), 'direction': np.int64(0), 'mission': Mission("One target is contained within the region from (1, 1) to (6, 6), one target is contained within the region from (16,16) to (21,21) and the other target is contained within the region from (40, 40) to (46, 46)."), 'location': (np.int64(1), np.int64(1))}, 2: {'image': array([[[1, 0, 0]]]), 'direction': np.int64(0), 'mission': Mission("One target is contained within the region from (1, 1) to (6, 6), one target is contained within the region from (16,16) to (21,21) and the other target is contained within the region from (40, 40) to (46, 46)."), 'location': (np.int64(1), np.int64(